# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/egeklc/flyrank-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*



### **Finding 1 - What Predicts Growth?**

**The claim:** Logistic regression model separates growing from declining pages with 71% holdout accuracy. Content age is the strongest negative signal, while days visible and recent impressions are the strongest positive signals.

**Where the label comes from:** The label is Trend direction which is derived from 30 day vs previous 30day impression change.

**Does the validation design carry the claim?** The paper's methodology states an 80/20 split but it doesn't specify whether that split is  grouped by brand or random row split. The data spans 57 brands across 341,701 pages. If the split is random row split, pages from the same brand can appear in both training and test sets, which is a leakage risk.

**Methodology question:** Was the 80/20 split random or grouped by brand? Was the reported 71% holdout accuracy calculated on a genuinely independent test population?


### **Finding 2 - What Predicts Health?**

**The claim:** Random Forest feature importance identifies average position, impressions, and scroll depth as the strongest features for predicting Health Score.

**Where the label comes from:** The label is Health Score, constructed from impressions, position, CTR, and scroll depth.

**Does the validation design carry the claim?** The model is tested on a holdout set, so it can show that the model predicts Health Score. However, some of the features are already used to calculate Health Score. This means their high importance is expected and does not show that they independently cause better health. The paper also states that importance is descriptive rather than causal.

**Methodology question:** Are these features truly independent predictors of health, or are they important simply because they are already part of the Health Score?

In [1]:
!git clone https://github.com/egeklc/flyrank-internship.git

import os
os.chdir("flyrank-internship")
print(os.getcwd())

fatal: destination path 'flyrank-internship' already exists and is not an empty directory.
/content/flyrank-internship


In [2]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining"] = df["trend_direction"].str.lower().eq("down").astype(int)

print("Number of pages: ", len(df))
print("Number of clients: ", df["client_id"].nunique())

Number of pages:  30000
Number of clients:  32


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [3]:
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score


target_col = "is_declining"
group_col = "client_id"

excluded_cols = ['content_id', 'client_id',
                 "trend_direction", "trend_pct", "is_declining",
                 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d',
                 'clicks_prev_30d', 'sessions_prev_30d'
                 ]

num_cols = [col for col in df.select_dtypes(include=["int64", "float64"]).columns
            if col not in excluded_cols
            ]

cat_cols = [col for col in df.select_dtypes(include=["object"]).columns
            if col not in excluded_cols
            ]

print(num_cols)
print(cat_cols)


feature_cols = num_cols + cat_cols
model_df = df[feature_cols + [target_col, group_col]].copy()


def build_random_forest_model(num_cols, cat_cols):

    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value=0))
    ])

    categorical_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ])

    model = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=300,
            max_depth=20,
            min_samples_split=15,
            min_samples_leaf=8,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ])

    return model

def precision_at_k(y_true, scores, k=50):
    order = np.argsort(scores)[::-1]
    top_k = order[:k]

    return y_true.iloc[top_k].mean()


['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'age_tier_order', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']
['competition_level', 'content_type', 'main_intent', 'provider_used', 'model_used', 'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'impression_tier', 'position_tier']


In [4]:
X = model_df[feature_cols]
y = model_df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

random_train_clients = set(model_df.loc[X_train.index, group_col])
random_test_clients = set(model_df.loc[X_test.index, group_col])
shared_random_clients = random_train_clients & random_test_clients

print("Random split")
print("Train shape:", X_train.shape, " | Test shape:", X_test.shape)
print("Train positive rate:", y_train.mean())
print("Test positive rate:", y_test.mean())
print("Shared clients:", len(shared_random_clients))

random_forest_random = build_random_forest_model(num_cols,cat_cols)

random_forest_random.fit(X_train, y_train)
random_pred = random_forest_random.predict_proba(X_test)[:, 1]

random_p50 = precision_at_k(y_test, random_pred)
random_roc_auc = roc_auc_score(y_test, random_pred)

Random split
Train shape: (22500, 34)  | Test shape: (7500, 34)
Train positive rate: 0.5420888888888888
Test positive rate: 0.542
Shared clients: 31


In [5]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)

train_idx, test_idx = next(gss.split(
    model_df[feature_cols],
    model_df[target_col],
    groups=model_df[group_col]
    )
)

X_train_grouped = model_df.iloc[train_idx][feature_cols]
X_test_grouped = model_df.iloc[test_idx][feature_cols]

y_train_grouped = model_df.iloc[train_idx][target_col]
y_test_grouped = model_df.iloc[test_idx][target_col]

groups_train = model_df.iloc[train_idx][group_col]
groups_test = model_df.iloc[test_idx][group_col]

print("Grouped split")
print("Train shape:", X_train_grouped.shape, " | Test shape:", X_test_grouped.shape)
print("Train clients:", groups_train.nunique())
print("Test clients:", groups_test.nunique())
print("Shared clients (overlap):", len(set(groups_train) & set(groups_test)))
print("Train positive rate:", y_train_grouped.mean())
print("Test positive rate:", y_test_grouped.mean())

random_forest_grouped = build_random_forest_model(num_cols, cat_cols)

random_forest_grouped.fit(X_train_grouped, y_train_grouped)
grouped_pred = random_forest_grouped.predict_proba(X_test_grouped)[:, 1]

grouped_p50 = precision_at_k(y_test_grouped, grouped_pred)
grouped_roc_auc = roc_auc_score(y_test_grouped, grouped_pred)

Grouped split
Train shape: (22885, 34)  | Test shape: (7115, 34)
Train clients: 24
Test clients: 8
Shared clients (overlap): 0
Train positive rate: 0.5500109241861482
Test positive rate: 0.516514406184118


In [6]:
comparison_df = pd.DataFrame({"Split Type":["Random Split", "Grouped Split (Honest)"],
                              "Precision@50":[random_p50, grouped_p50],
                              "ROC-AUC":[random_roc_auc, grouped_roc_auc]
                              })
print(comparison_df)

               Split Type  Precision@50   ROC-AUC
0            Random Split          1.00  0.775684
1  Grouped Split (Honest)          0.68  0.600047


The random split has 31 of 32 clients shared between training and test sets, so the model can learn client specific patterns it has already seen, which can boost the model performance. The model achieved 1.00 Precision@50 and 0.776 ROC-AUC on the test set with the random split.

The grouped split (honest) eliminates this overlap. In the grouped split 24 clients are used for training set and the remaining 8 clients are used for test set. The model achieved 0.68 Precision@50 and 0.60 ROC-AUC on the test set.

The difference in performance shows that the random split gives a higher and likely overly optimistic performance. The grouped split gives a more realistic estimate of how the model performs on unseen clients.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [9]:
print("Label-derived columns:",
      [col for col in ["is_declining", "trend_direction", "trend_pct"]
       if col in feature_cols])

print("Label-window columns:",
      [col for col in [
          "impressions_last_30d",
          "clicks_last_30d",
          "sessions_last_30d",
          "impressions_prev_30d",
          "clicks_prev_30d",
          "sessions_prev_30d"
      ] if col in feature_cols])

print("ID columns:",
      [col for col in ["content_id", "client_id"]
       if col in feature_cols])

Label-derived columns: []
Label-window columns: []
ID columns: []


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Claim:**
The Random Forest model predicts declining pages better than the Week 4 Baseline model and identifies the most important factors for predicting page decline.

**Safer Claim:**
The Random Forest model achieved 0.68 Precision@50 and 0.60 ROC-AUC using a client-grouped split. These results suggest that the model can help rank pages by their likelihood of decline for unseen clients. Feature importance shows which features contribute most to the model's predictions, but does not show causation. The resulting score can be used to prioritize pages for review.

## Self-check

Before you submit, confirm each line honestly:

- [+] Every section above is filled — markdown thinking AND the code that backs it
- [+] The notebook runs top to bottom with no errors (Runtime → Run all)
- [+] No client names, URLs, or private queries anywhere
- [+] My claims use careful words: observed, measured, directional, decision-support
- [+] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.